# Análise exploratória | MRBR | vw_ds_pro_mrbr_bloqueado_devido_desvios

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`` |
| Ambiente | DEV |
| Objeto | View |
| Colunas do DESCRIBE | 70 |
| Tipos | decimal(18,2), decimal(18,3), int, string |
| Clustering declarado | Não informado no CSV |
| Centro | `cod_centro` |
| Chave candidata | `num_doc_faturamento` + `num_exercicio` + `num_item_fatura` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_MRBR_2.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | 2 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |
| 🟡 MÉDIA | View: conferir comentário no DESCRIBE EXTENDED; DETAIL/HISTORY não executados. |
| 🟡 MÉDIA | Há campos documentados como sem fonte/indisponíveis; verificar preenchimento sem classificá-los automaticamente como erro. |
| 🟡 MÉDIA | Há datas STRING: medir formato e sentinelas antes de converter. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`;


### 1.1. DETAIL/HISTORY da view

Não executados: estes comandos não são válidos para view. Consultar o comentário da view na seção 1.


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `num_doc_faturamento`) AS distintos_num_doc_faturamento, COUNT(DISTINCT `cod_material`) AS distintos_cod_material, COUNT(DISTINCT `num_item_fatura`) AS distintos_num_item_fatura, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`;


## 3. Distribuição por centro

Mede volumes e sinaliza centros entre 10 mil e 300 mil registros, sem criar cenários.


In [0]:
SELECT `cod_centro` AS centro, COUNT(*) AS linhas, CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE 'AVALIAR VOLUME' END AS avaliacao FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_centro` ORDER BY linhas DESC;


## 3.1. Centros 4128 e 4014

Confirma a presença e o formato literal dos centros padrão; ainda não filtra os demais testes.


In [0]:
SELECT `cod_centro` AS centro_literal, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) AS tamanho, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` WHERE TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0*(4128|4014)$' GROUP BY `cod_centro`, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) ORDER BY centro_literal;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'num_doc_faturamento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio + num_item_fatura' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio + num_item_fatura + num_seq_classcont' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`, `num_seq_classcont` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) k UNION ALL SELECT 'num_doc_faturamento + num_exercicio + num_item_fatura + num_doc_compras + num_item_pedido' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`, `num_doc_compras`, `num_item_pedido` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`num_doc_faturamento`, t.`num_exercicio`, t.`num_item_fatura`, md5(to_json(struct(t.`num_exercicio`, t.`num_exerc_periodo_atual`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`num_doc_referencia`, t.`num_doc_compras`, t.`cod_centro`, t.`cod_material`, t.`num_item_pedido`, t.`cod_grupo_compradores`, t.`qt_pedido`, t.`sg_um_pedido`, t.`vl_montante`, t.`sg_moeda`, t.`vl_diferenca`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`num_item_fatura`, t.`num_seq_classcont`, t.`num_item_doc_referencia`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`tp_condicao`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`qt_diferenca_1`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_qtd`, t.`ind_bloq_qualidade_icone`, t.`ind_bloq_qtd_icone`, t.`ind_bloq_qualidade`, t.`ind_bloq_preco_icone`, t.`ind_bloq_qupp_icone`, t.`ind_bloq_qupp`, t.`ind_bloq_preco`, t.`ind_bloq_montante_icone`, t.`ind_bloq_montante_1`, t.`ind_bloq_montante_2`, t.`ind_bloq_manual_icone`, t.`ind_bloq_manual_1`, t.`ind_bloq_manual_2`, t.`ind_bloq_data_icone`, t.`ind_bloq_data`, t.`num_linha_servico`, t.`num_item_contrato_central`, t.`num_item_contrato_superior`, t.`qt_diferenca_dias`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`ind_debito_posterior`, t.`dt_conversao`, t.`num_contrato_central`, t.`num_contrato_basico`, t.`ind_custo_acessorio`, t.`cod_chave_controle_sasq`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` t), d AS (SELECT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`num_exercicio` AS STRING)) AS `dist_num_exercicio`, COUNT(DISTINCT CAST(`num_exerc_periodo_atual` AS STRING)) AS `dist_num_exerc_periodo_atual`, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS `dist_cod_empresa`, COUNT(DISTINCT CAST(`num_doc_faturamento` AS STRING)) AS `dist_num_doc_faturamento`, COUNT(DISTINCT CAST(`cod_emissor_fatura` AS STRING)) AS `dist_cod_emissor_fatura`, COUNT(DISTINCT CAST(`dt_base_prazo_pgto` AS STRING)) AS `dist_dt_base_prazo_pgto`, COUNT(DISTINCT CAST(`dt_lancamento` AS STRING)) AS `dist_dt_lancamento`, COUNT(DISTINCT CAST(`qt_dias_desconto_1` AS STRING)) AS `dist_qt_dias_desconto_1`, COUNT(DISTINCT CAST(`num_doc_referencia` AS STRING)) AS `dist_num_doc_referencia`, COUNT(DISTINCT CAST(`num_doc_compras` AS STRING)) AS `dist_num_doc_compras`, COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS `dist_cod_centro`, COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS `dist_cod_material` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_num_exercicio` > 1) AS `chaves_variam_num_exercicio`, COUNT_IF(`dist_num_exerc_periodo_atual` > 1) AS `chaves_variam_num_exerc_periodo_atual`, COUNT_IF(`dist_cod_empresa` > 1) AS `chaves_variam_cod_empresa`, COUNT_IF(`dist_num_doc_faturamento` > 1) AS `chaves_variam_num_doc_faturamento`, COUNT_IF(`dist_cod_emissor_fatura` > 1) AS `chaves_variam_cod_emissor_fatura`, COUNT_IF(`dist_dt_base_prazo_pgto` > 1) AS `chaves_variam_dt_base_prazo_pgto`, COUNT_IF(`dist_dt_lancamento` > 1) AS `chaves_variam_dt_lancamento`, COUNT_IF(`dist_qt_dias_desconto_1` > 1) AS `chaves_variam_qt_dias_desconto_1`, COUNT_IF(`dist_num_doc_referencia` > 1) AS `chaves_variam_num_doc_referencia`, COUNT_IF(`dist_num_doc_compras` > 1) AS `chaves_variam_num_doc_compras`, COUNT_IF(`dist_cod_centro` > 1) AS `chaves_variam_cod_centro`, COUNT_IF(`dist_cod_material` > 1) AS `chaves_variam_cod_material` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(24, 'num_exercicio', 'string', c0_n, c0_v, c0_z, 'num_exerc_periodo_atual', 'string', c1_n, c1_v, c1_z, 'cod_empresa', 'string', c2_n, c2_v, c2_z, 'nm_fornecedor', 'string', c3_n, c3_v, c3_z, 'num_doc_faturamento', 'string', c4_n, c4_v, c4_z, 'cod_emissor_fatura', 'string', c5_n, c5_v, c5_z, 'dt_base_prazo_pgto', 'string', c6_n, c6_v, c6_z, 'dt_lancamento', 'string', c7_n, c7_v, c7_z, 'qt_dias_desconto_1', 'int', c8_n, c8_v, c8_z, 'num_doc_referencia', 'string', c9_n, c9_v, c9_z, 'num_doc_compras', 'string', c10_n, c10_v, c10_z, 'cod_centro', 'string', c11_n, c11_v, c11_z, 'cod_material', 'string', c12_n, c12_v, c12_z, 'num_item_pedido', 'string', c13_n, c13_v, c13_z, 'cod_grupo_compradores', 'string', c14_n, c14_v, c14_z, 'qt_pedido', 'decimal(18,3)', c15_n, c15_v, c15_z, 'sg_um_pedido', 'string', c16_n, c16_v, c16_z, 'vl_montante', 'decimal(18,2)', c17_n, c17_v, c17_z, 'sg_moeda', 'string', c18_n, c18_v, c18_z, 'vl_diferenca', 'decimal(18,2)', c19_n, c19_v, c19_z, 'sg_moeda_interna', 'string', c20_n, c20_v, c20_z, 'tp_documento', 'string', c21_n, c21_v, c21_z, 'cod_bloq_pagamento_log', 'string', c22_n, c22_v, c22_z, 'nm_usuario', 'string', c23_n, c23_v, c23_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_exercicio` IS NULL) AS c0_n, COUNT_IF(`num_exercicio` IS NOT NULL AND LOWER(TRIM(`num_exercicio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`num_exerc_periodo_atual` IS NULL) AS c1_n, COUNT_IF(`num_exerc_periodo_atual` IS NOT NULL AND LOWER(TRIM(`num_exerc_periodo_atual`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`num_exerc_periodo_atual`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`cod_empresa` IS NULL) AS c2_n, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`nm_fornecedor` IS NULL) AS c3_n, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c3_v, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c3_z, COUNT_IF(`num_doc_faturamento` IS NULL) AS c4_n, COUNT_IF(`num_doc_faturamento` IS NOT NULL AND LOWER(TRIM(`num_doc_faturamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c4_v, COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+([.,]0+)?$') AS c4_z, COUNT_IF(`cod_emissor_fatura` IS NULL) AS c5_n, COUNT_IF(`cod_emissor_fatura` IS NOT NULL AND LOWER(TRIM(`cod_emissor_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`dt_base_prazo_pgto` IS NULL) AS c6_n, COUNT_IF(`dt_base_prazo_pgto` IS NOT NULL AND LOWER(TRIM(`dt_base_prazo_pgto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c6_v, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^0+([.,]0+)?$') AS c6_z, COUNT_IF(`dt_lancamento` IS NULL) AS c7_n, COUNT_IF(`dt_lancamento` IS NOT NULL AND LOWER(TRIM(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c7_v, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^0+([.,]0+)?$') AS c7_z, COUNT_IF(`qt_dias_desconto_1` IS NULL) AS c8_n, CAST(0 AS BIGINT) AS c8_v, COUNT_IF(`qt_dias_desconto_1` = 0) AS c8_z, COUNT_IF(`num_doc_referencia` IS NULL) AS c9_n, COUNT_IF(`num_doc_referencia` IS NOT NULL AND LOWER(TRIM(`num_doc_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c9_v, COUNT_IF(TRIM(`num_doc_referencia`) RLIKE '^0+([.,]0+)?$') AS c9_z, COUNT_IF(`num_doc_compras` IS NULL) AS c10_n, COUNT_IF(`num_doc_compras` IS NOT NULL AND LOWER(TRIM(`num_doc_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`num_doc_compras`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`cod_centro` IS NULL) AS c11_n, COUNT_IF(`cod_centro` IS NOT NULL AND LOWER(TRIM(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS c11_z, COUNT_IF(`cod_material` IS NULL) AS c12_n, COUNT_IF(`cod_material` IS NOT NULL AND LOWER(TRIM(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c12_v, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+([.,]0+)?$') AS c12_z, COUNT_IF(`num_item_pedido` IS NULL) AS c13_n, COUNT_IF(`num_item_pedido` IS NOT NULL AND LOWER(TRIM(`num_item_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c13_v, COUNT_IF(TRIM(`num_item_pedido`) RLIKE '^0+([.,]0+)?$') AS c13_z, COUNT_IF(`cod_grupo_compradores` IS NULL) AS c14_n, COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND LOWER(TRIM(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c14_v, COUNT_IF(TRIM(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$') AS c14_z, COUNT_IF(`qt_pedido` IS NULL) AS c15_n, CAST(0 AS BIGINT) AS c15_v, COUNT_IF(`qt_pedido` = 0) AS c15_z, COUNT_IF(`sg_um_pedido` IS NULL) AS c16_n, COUNT_IF(`sg_um_pedido` IS NOT NULL AND LOWER(TRIM(`sg_um_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c16_v, COUNT_IF(TRIM(`sg_um_pedido`) RLIKE '^0+([.,]0+)?$') AS c16_z, COUNT_IF(`vl_montante` IS NULL) AS c17_n, CAST(0 AS BIGINT) AS c17_v, COUNT_IF(`vl_montante` = 0) AS c17_z, COUNT_IF(`sg_moeda` IS NULL) AS c18_n, COUNT_IF(`sg_moeda` IS NOT NULL AND LOWER(TRIM(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c18_v, COUNT_IF(TRIM(`sg_moeda`) RLIKE '^0+([.,]0+)?$') AS c18_z, COUNT_IF(`vl_diferenca` IS NULL) AS c19_n, CAST(0 AS BIGINT) AS c19_v, COUNT_IF(`vl_diferenca` = 0) AS c19_z, COUNT_IF(`sg_moeda_interna` IS NULL) AS c20_n, COUNT_IF(`sg_moeda_interna` IS NOT NULL AND LOWER(TRIM(`sg_moeda_interna`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c20_v, COUNT_IF(TRIM(`sg_moeda_interna`) RLIKE '^0+([.,]0+)?$') AS c20_z, COUNT_IF(`tp_documento` IS NULL) AS c21_n, COUNT_IF(`tp_documento` IS NOT NULL AND LOWER(TRIM(`tp_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c21_v, COUNT_IF(TRIM(`tp_documento`) RLIKE '^0+([.,]0+)?$') AS c21_z, COUNT_IF(`cod_bloq_pagamento_log` IS NULL) AS c22_n, COUNT_IF(`cod_bloq_pagamento_log` IS NOT NULL AND LOWER(TRIM(`cod_bloq_pagamento_log`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c22_v, COUNT_IF(TRIM(`cod_bloq_pagamento_log`) RLIKE '^0+([.,]0+)?$') AS c22_z, COUNT_IF(`nm_usuario` IS NULL) AS c23_n, COUNT_IF(`nm_usuario` IS NOT NULL AND LOWER(TRIM(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c23_v, COUNT_IF(TRIM(`nm_usuario`) RLIKE '^0+([.,]0+)?$') AS c23_z FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) a UNION ALL SELECT stack(24, 'num_item_fatura', 'string', c24_n, c24_v, c24_z, 'num_seq_classcont', 'int', c25_n, c25_v, c25_z, 'num_item_doc_referencia', 'string', c26_n, c26_v, c26_z, 'cod_bloqueio_pgto', 'string', c27_n, c27_v, c27_z, 'sg_upp', 'string', c28_n, c28_v, c28_z, 'tp_condicao', 'string', c29_n, c29_v, c29_z, 'pct_desconto_2', 'string', c30_n, c30_v, c30_z, 'pct_desconto_1', 'string', c31_n, c31_v, c31_z, 'vl_taxa_cambio', 'string', c32_n, c32_v, c32_z, 'st_fatura', 'string', c33_n, c33_v, c33_z, 'qt_em_upp', 'decimal(18,3)', c34_n, c34_v, c34_z, 'qt_diferenca', 'decimal(18,3)', c35_n, c35_v, c35_z, 'qt_diferenca_1', 'decimal(18,3)', c36_n, c36_v, c36_z, 'cod_determ_preco', 'string', c37_n, c37_v, c37_z, 'cod_pais_destino', 'string', c38_n, c38_v, c38_z, 'cod_pais_procedencia', 'string', c39_n, c39_v, c39_z, 'ind_nota_existente', 'string', c40_n, c40_v, c40_z, 'num_pacote', 'string', c41_n, c41_v, c41_z, 'ind_bloq_qtd', 'string', c42_n, c42_v, c42_z, 'ind_bloq_qualidade_icone', 'string', c43_n, c43_v, c43_z, 'ind_bloq_qtd_icone', 'string', c44_n, c44_v, c44_z, 'ind_bloq_qualidade', 'string', c45_n, c45_v, c45_z, 'ind_bloq_preco_icone', 'string', c46_n, c46_v, c46_z, 'ind_bloq_qupp_icone', 'string', c47_n, c47_v, c47_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_item_fatura` IS NULL) AS c24_n, COUNT_IF(`num_item_fatura` IS NOT NULL AND LOWER(TRIM(`num_item_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c24_v, COUNT_IF(TRIM(`num_item_fatura`) RLIKE '^0+([.,]0+)?$') AS c24_z, COUNT_IF(`num_seq_classcont` IS NULL) AS c25_n, CAST(0 AS BIGINT) AS c25_v, COUNT_IF(`num_seq_classcont` = 0) AS c25_z, COUNT_IF(`num_item_doc_referencia` IS NULL) AS c26_n, COUNT_IF(`num_item_doc_referencia` IS NOT NULL AND LOWER(TRIM(`num_item_doc_referencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c26_v, COUNT_IF(TRIM(`num_item_doc_referencia`) RLIKE '^0+([.,]0+)?$') AS c26_z, COUNT_IF(`cod_bloqueio_pgto` IS NULL) AS c27_n, COUNT_IF(`cod_bloqueio_pgto` IS NOT NULL AND LOWER(TRIM(`cod_bloqueio_pgto`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c27_v, COUNT_IF(TRIM(`cod_bloqueio_pgto`) RLIKE '^0+([.,]0+)?$') AS c27_z, COUNT_IF(`sg_upp` IS NULL) AS c28_n, COUNT_IF(`sg_upp` IS NOT NULL AND LOWER(TRIM(`sg_upp`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c28_v, COUNT_IF(TRIM(`sg_upp`) RLIKE '^0+([.,]0+)?$') AS c28_z, COUNT_IF(`tp_condicao` IS NULL) AS c29_n, COUNT_IF(`tp_condicao` IS NOT NULL AND LOWER(TRIM(`tp_condicao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c29_v, COUNT_IF(TRIM(`tp_condicao`) RLIKE '^0+([.,]0+)?$') AS c29_z, COUNT_IF(`pct_desconto_2` IS NULL) AS c30_n, COUNT_IF(`pct_desconto_2` IS NOT NULL AND LOWER(TRIM(`pct_desconto_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c30_v, COUNT_IF(TRIM(`pct_desconto_2`) RLIKE '^0+([.,]0+)?$') AS c30_z, COUNT_IF(`pct_desconto_1` IS NULL) AS c31_n, COUNT_IF(`pct_desconto_1` IS NOT NULL AND LOWER(TRIM(`pct_desconto_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c31_v, COUNT_IF(TRIM(`pct_desconto_1`) RLIKE '^0+([.,]0+)?$') AS c31_z, COUNT_IF(`vl_taxa_cambio` IS NULL) AS c32_n, COUNT_IF(`vl_taxa_cambio` IS NOT NULL AND LOWER(TRIM(`vl_taxa_cambio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c32_v, COUNT_IF(TRIM(`vl_taxa_cambio`) RLIKE '^0+([.,]0+)?$') AS c32_z, COUNT_IF(`st_fatura` IS NULL) AS c33_n, COUNT_IF(`st_fatura` IS NOT NULL AND LOWER(TRIM(`st_fatura`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c33_v, COUNT_IF(TRIM(`st_fatura`) RLIKE '^0+([.,]0+)?$') AS c33_z, COUNT_IF(`qt_em_upp` IS NULL) AS c34_n, CAST(0 AS BIGINT) AS c34_v, COUNT_IF(`qt_em_upp` = 0) AS c34_z, COUNT_IF(`qt_diferenca` IS NULL) AS c35_n, CAST(0 AS BIGINT) AS c35_v, COUNT_IF(`qt_diferenca` = 0) AS c35_z, COUNT_IF(`qt_diferenca_1` IS NULL) AS c36_n, CAST(0 AS BIGINT) AS c36_v, COUNT_IF(`qt_diferenca_1` = 0) AS c36_z, COUNT_IF(`cod_determ_preco` IS NULL) AS c37_n, COUNT_IF(`cod_determ_preco` IS NOT NULL AND LOWER(TRIM(`cod_determ_preco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c37_v, COUNT_IF(TRIM(`cod_determ_preco`) RLIKE '^0+([.,]0+)?$') AS c37_z, COUNT_IF(`cod_pais_destino` IS NULL) AS c38_n, COUNT_IF(`cod_pais_destino` IS NOT NULL AND LOWER(TRIM(`cod_pais_destino`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c38_v, COUNT_IF(TRIM(`cod_pais_destino`) RLIKE '^0+([.,]0+)?$') AS c38_z, COUNT_IF(`cod_pais_procedencia` IS NULL) AS c39_n, COUNT_IF(`cod_pais_procedencia` IS NOT NULL AND LOWER(TRIM(`cod_pais_procedencia`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c39_v, COUNT_IF(TRIM(`cod_pais_procedencia`) RLIKE '^0+([.,]0+)?$') AS c39_z, COUNT_IF(`ind_nota_existente` IS NULL) AS c40_n, COUNT_IF(`ind_nota_existente` IS NOT NULL AND LOWER(TRIM(`ind_nota_existente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c40_v, COUNT_IF(TRIM(`ind_nota_existente`) RLIKE '^0+([.,]0+)?$') AS c40_z, COUNT_IF(`num_pacote` IS NULL) AS c41_n, COUNT_IF(`num_pacote` IS NOT NULL AND LOWER(TRIM(`num_pacote`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c41_v, COUNT_IF(TRIM(`num_pacote`) RLIKE '^0+([.,]0+)?$') AS c41_z, COUNT_IF(`ind_bloq_qtd` IS NULL) AS c42_n, COUNT_IF(`ind_bloq_qtd` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qtd`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c42_v, COUNT_IF(TRIM(`ind_bloq_qtd`) RLIKE '^0+([.,]0+)?$') AS c42_z, COUNT_IF(`ind_bloq_qualidade_icone` IS NULL) AS c43_n, COUNT_IF(`ind_bloq_qualidade_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qualidade_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c43_v, COUNT_IF(TRIM(`ind_bloq_qualidade_icone`) RLIKE '^0+([.,]0+)?$') AS c43_z, COUNT_IF(`ind_bloq_qtd_icone` IS NULL) AS c44_n, COUNT_IF(`ind_bloq_qtd_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qtd_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c44_v, COUNT_IF(TRIM(`ind_bloq_qtd_icone`) RLIKE '^0+([.,]0+)?$') AS c44_z, COUNT_IF(`ind_bloq_qualidade` IS NULL) AS c45_n, COUNT_IF(`ind_bloq_qualidade` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qualidade`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c45_v, COUNT_IF(TRIM(`ind_bloq_qualidade`) RLIKE '^0+([.,]0+)?$') AS c45_z, COUNT_IF(`ind_bloq_preco_icone` IS NULL) AS c46_n, COUNT_IF(`ind_bloq_preco_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_preco_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c46_v, COUNT_IF(TRIM(`ind_bloq_preco_icone`) RLIKE '^0+([.,]0+)?$') AS c46_z, COUNT_IF(`ind_bloq_qupp_icone` IS NULL) AS c47_n, COUNT_IF(`ind_bloq_qupp_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qupp_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c47_v, COUNT_IF(TRIM(`ind_bloq_qupp_icone`) RLIKE '^0+([.,]0+)?$') AS c47_z FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) a UNION ALL SELECT stack(22, 'ind_bloq_qupp', 'string', c48_n, c48_v, c48_z, 'ind_bloq_preco', 'string', c49_n, c49_v, c49_z, 'ind_bloq_montante_icone', 'string', c50_n, c50_v, c50_z, 'ind_bloq_montante_1', 'string', c51_n, c51_v, c51_z, 'ind_bloq_montante_2', 'string', c52_n, c52_v, c52_z, 'ind_bloq_manual_icone', 'string', c53_n, c53_v, c53_z, 'ind_bloq_manual_1', 'string', c54_n, c54_v, c54_z, 'ind_bloq_manual_2', 'string', c55_n, c55_v, c55_z, 'ind_bloq_data_icone', 'string', c56_n, c56_v, c56_z, 'ind_bloq_data', 'string', c57_n, c57_v, c57_z, 'num_linha_servico', 'string', c58_n, c58_v, c58_z, 'num_item_contrato_central', 'string', c59_n, c59_v, c59_z, 'num_item_contrato_superior', 'string', c60_n, c60_v, c60_z, 'qt_diferenca_dias', 'string', c61_n, c61_v, c61_z, 'qt_dias_liquido', 'int', c62_n, c62_v, c62_z, 'qt_dias_desconto_2', 'int', c63_n, c63_v, c63_z, 'ind_debito_posterior', 'string', c64_n, c64_v, c64_z, 'dt_conversao', 'string', c65_n, c65_v, c65_z, 'num_contrato_central', 'string', c66_n, c66_v, c66_z, 'num_contrato_basico', 'string', c67_n, c67_v, c67_z, 'ind_custo_acessorio', 'string', c68_n, c68_v, c68_z, 'cod_chave_controle_sasq', 'string', c69_n, c69_v, c69_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`ind_bloq_qupp` IS NULL) AS c48_n, COUNT_IF(`ind_bloq_qupp` IS NOT NULL AND LOWER(TRIM(`ind_bloq_qupp`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c48_v, COUNT_IF(TRIM(`ind_bloq_qupp`) RLIKE '^0+([.,]0+)?$') AS c48_z, COUNT_IF(`ind_bloq_preco` IS NULL) AS c49_n, COUNT_IF(`ind_bloq_preco` IS NOT NULL AND LOWER(TRIM(`ind_bloq_preco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c49_v, COUNT_IF(TRIM(`ind_bloq_preco`) RLIKE '^0+([.,]0+)?$') AS c49_z, COUNT_IF(`ind_bloq_montante_icone` IS NULL) AS c50_n, COUNT_IF(`ind_bloq_montante_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c50_v, COUNT_IF(TRIM(`ind_bloq_montante_icone`) RLIKE '^0+([.,]0+)?$') AS c50_z, COUNT_IF(`ind_bloq_montante_1` IS NULL) AS c51_n, COUNT_IF(`ind_bloq_montante_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c51_v, COUNT_IF(TRIM(`ind_bloq_montante_1`) RLIKE '^0+([.,]0+)?$') AS c51_z, COUNT_IF(`ind_bloq_montante_2` IS NULL) AS c52_n, COUNT_IF(`ind_bloq_montante_2` IS NOT NULL AND LOWER(TRIM(`ind_bloq_montante_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c52_v, COUNT_IF(TRIM(`ind_bloq_montante_2`) RLIKE '^0+([.,]0+)?$') AS c52_z, COUNT_IF(`ind_bloq_manual_icone` IS NULL) AS c53_n, COUNT_IF(`ind_bloq_manual_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c53_v, COUNT_IF(TRIM(`ind_bloq_manual_icone`) RLIKE '^0+([.,]0+)?$') AS c53_z, COUNT_IF(`ind_bloq_manual_1` IS NULL) AS c54_n, COUNT_IF(`ind_bloq_manual_1` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c54_v, COUNT_IF(TRIM(`ind_bloq_manual_1`) RLIKE '^0+([.,]0+)?$') AS c54_z, COUNT_IF(`ind_bloq_manual_2` IS NULL) AS c55_n, COUNT_IF(`ind_bloq_manual_2` IS NOT NULL AND LOWER(TRIM(`ind_bloq_manual_2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c55_v, COUNT_IF(TRIM(`ind_bloq_manual_2`) RLIKE '^0+([.,]0+)?$') AS c55_z, COUNT_IF(`ind_bloq_data_icone` IS NULL) AS c56_n, COUNT_IF(`ind_bloq_data_icone` IS NOT NULL AND LOWER(TRIM(`ind_bloq_data_icone`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c56_v, COUNT_IF(TRIM(`ind_bloq_data_icone`) RLIKE '^0+([.,]0+)?$') AS c56_z, COUNT_IF(`ind_bloq_data` IS NULL) AS c57_n, COUNT_IF(`ind_bloq_data` IS NOT NULL AND LOWER(TRIM(`ind_bloq_data`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c57_v, COUNT_IF(TRIM(`ind_bloq_data`) RLIKE '^0+([.,]0+)?$') AS c57_z, COUNT_IF(`num_linha_servico` IS NULL) AS c58_n, COUNT_IF(`num_linha_servico` IS NOT NULL AND LOWER(TRIM(`num_linha_servico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c58_v, COUNT_IF(TRIM(`num_linha_servico`) RLIKE '^0+([.,]0+)?$') AS c58_z, COUNT_IF(`num_item_contrato_central` IS NULL) AS c59_n, COUNT_IF(`num_item_contrato_central` IS NOT NULL AND LOWER(TRIM(`num_item_contrato_central`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c59_v, COUNT_IF(TRIM(`num_item_contrato_central`) RLIKE '^0+([.,]0+)?$') AS c59_z, COUNT_IF(`num_item_contrato_superior` IS NULL) AS c60_n, COUNT_IF(`num_item_contrato_superior` IS NOT NULL AND LOWER(TRIM(`num_item_contrato_superior`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c60_v, COUNT_IF(TRIM(`num_item_contrato_superior`) RLIKE '^0+([.,]0+)?$') AS c60_z, COUNT_IF(`qt_diferenca_dias` IS NULL) AS c61_n, COUNT_IF(`qt_diferenca_dias` IS NOT NULL AND LOWER(TRIM(`qt_diferenca_dias`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c61_v, COUNT_IF(TRIM(`qt_diferenca_dias`) RLIKE '^0+([.,]0+)?$') AS c61_z, COUNT_IF(`qt_dias_liquido` IS NULL) AS c62_n, CAST(0 AS BIGINT) AS c62_v, COUNT_IF(`qt_dias_liquido` = 0) AS c62_z, COUNT_IF(`qt_dias_desconto_2` IS NULL) AS c63_n, CAST(0 AS BIGINT) AS c63_v, COUNT_IF(`qt_dias_desconto_2` = 0) AS c63_z, COUNT_IF(`ind_debito_posterior` IS NULL) AS c64_n, COUNT_IF(`ind_debito_posterior` IS NOT NULL AND LOWER(TRIM(`ind_debito_posterior`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c64_v, COUNT_IF(TRIM(`ind_debito_posterior`) RLIKE '^0+([.,]0+)?$') AS c64_z, COUNT_IF(`dt_conversao` IS NULL) AS c65_n, COUNT_IF(`dt_conversao` IS NOT NULL AND LOWER(TRIM(`dt_conversao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c65_v, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^0+([.,]0+)?$') AS c65_z, COUNT_IF(`num_contrato_central` IS NULL) AS c66_n, COUNT_IF(`num_contrato_central` IS NOT NULL AND LOWER(TRIM(`num_contrato_central`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c66_v, COUNT_IF(TRIM(`num_contrato_central`) RLIKE '^0+([.,]0+)?$') AS c66_z, COUNT_IF(`num_contrato_basico` IS NULL) AS c67_n, COUNT_IF(`num_contrato_basico` IS NOT NULL AND LOWER(TRIM(`num_contrato_basico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c67_v, COUNT_IF(TRIM(`num_contrato_basico`) RLIKE '^0+([.,]0+)?$') AS c67_z, COUNT_IF(`ind_custo_acessorio` IS NULL) AS c68_n, COUNT_IF(`ind_custo_acessorio` IS NOT NULL AND LOWER(TRIM(`ind_custo_acessorio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c68_v, COUNT_IF(TRIM(`ind_custo_acessorio`) RLIKE '^0+([.,]0+)?$') AS c68_z, COUNT_IF(`cod_chave_controle_sasq` IS NULL) AS c69_n, COUNT_IF(`cod_chave_controle_sasq` IS NOT NULL AND LOWER(TRIM(`cod_chave_controle_sasq`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c69_v, COUNT_IF(TRIM(`cod_chave_controle_sasq`) RLIKE '^0+([.,]0+)?$') AS c69_z FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte ou sem preenchimento declarado

Confere o comportamento documentado; 100% nulo não é automaticamente erro de carga.


In [0]:
WITH a AS (SELECT 'st_fatura' AS coluna, COUNT(*) AS total, COUNT_IF(`st_fatura` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'qt_diferenca' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'qt_diferenca_1' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca_1` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'cod_determ_preco' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_determ_preco` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_nota_existente' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_nota_existente` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_pacote' AS coluna, COUNT(*) AS total, COUNT_IF(`num_pacote` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_qualidade_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qualidade_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_qtd_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qtd_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_preco_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_preco_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_qupp_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_qupp_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_montante_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_montante_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_manual_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_manual_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'ind_bloq_data_icone' AS coluna, COUNT(*) AS total, COUNT_IF(`ind_bloq_data_icone` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_linha_servico' AS coluna, COUNT(*) AS total, COUNT_IF(`num_linha_servico` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_item_contrato_central' AS coluna, COUNT(*) AS total, COUNT_IF(`num_item_contrato_central` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'qt_diferenca_dias' AS coluna, COUNT(*) AS total, COUNT_IF(`qt_diferenca_dias` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_contrato_central' AS coluna, COUNT(*) AS total, COUNT_IF(`num_contrato_central` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'cod_chave_controle_sasq' AS coluna, COUNT(*) AS total, COUNT_IF(`cod_chave_controle_sasq` IS NULL) AS nulos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) SELECT coluna, total, nulos, CASE WHEN total = nulos THEN 'CONFIRMADO NO RECORTE' ELSE 'HA PREENCHIMENTO: INVESTIGAR' END AS leitura FROM a ORDER BY coluna;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `num_doc_faturamento`) AS d0, COUNT(DISTINCT `cod_material`) AS d1, COUNT(DISTINCT `num_item_fatura`) AS d2, COUNT(DISTINCT `cod_empresa`) AS d3, COUNT(DISTINCT `cod_emissor_fatura`) AS d4, COUNT(DISTINCT `cod_centro`) AS d5, COUNT(DISTINCT `cod_grupo_compradores`) AS d6, COUNT(DISTINCT `sg_um_pedido`) AS d7, COUNT(DISTINCT `sg_moeda`) AS d8 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(9, 'num_doc_faturamento', d0, 'cod_material', d1, 'num_item_fatura', d2, 'cod_empresa', d3, 'cod_emissor_fatura', d4, 'cod_centro', d5, 'cod_grupo_compradores', d6, 'sg_um_pedido', d7, 'sg_moeda', d8) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_empresa` UNION ALL SELECT 'cod_emissor_fatura' AS coluna, CAST(`cod_emissor_fatura` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_emissor_fatura` UNION ALL SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_centro` UNION ALL SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_grupo_compradores` UNION ALL SELECT 'sg_um_pedido' AS coluna, CAST(`sg_um_pedido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `sg_um_pedido` UNION ALL SELECT 'sg_moeda' AS coluna, CAST(`sg_moeda` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `sg_moeda` UNION ALL SELECT 'sg_moeda_interna' AS coluna, CAST(`sg_moeda_interna` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `sg_moeda_interna` UNION ALL SELECT 'tp_documento' AS coluna, CAST(`tp_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `tp_documento`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

Mede os estados observados, sem converter indicadores STRING em booleanos por suposição.


In [0]:
WITH a AS (SELECT 'tp_documento' AS coluna, CAST(`tp_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `tp_documento` UNION ALL SELECT 'cod_bloq_pagamento_log' AS coluna, CAST(`cod_bloq_pagamento_log` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_bloq_pagamento_log` UNION ALL SELECT 'cod_bloqueio_pgto' AS coluna, CAST(`cod_bloqueio_pgto` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_bloqueio_pgto` UNION ALL SELECT 'tp_condicao' AS coluna, CAST(`tp_condicao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `tp_condicao` UNION ALL SELECT 'st_fatura' AS coluna, CAST(`st_fatura` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `st_fatura` UNION ALL SELECT 'ind_nota_existente' AS coluna, CAST(`ind_nota_existente` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_nota_existente` UNION ALL SELECT 'ind_bloq_qtd' AS coluna, CAST(`ind_bloq_qtd` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_qtd` UNION ALL SELECT 'ind_bloq_qualidade_icone' AS coluna, CAST(`ind_bloq_qualidade_icone` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_qualidade_icone` UNION ALL SELECT 'ind_bloq_qtd_icone' AS coluna, CAST(`ind_bloq_qtd_icone` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_qtd_icone` UNION ALL SELECT 'ind_bloq_qualidade' AS coluna, CAST(`ind_bloq_qualidade` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_qualidade` UNION ALL SELECT 'ind_bloq_preco_icone' AS coluna, CAST(`ind_bloq_preco_icone` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_preco_icone` UNION ALL SELECT 'ind_bloq_qupp_icone' AS coluna, CAST(`ind_bloq_qupp_icone` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_qupp_icone` UNION ALL SELECT 'ind_bloq_qupp' AS coluna, CAST(`ind_bloq_qupp` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_qupp` UNION ALL SELECT 'ind_bloq_preco' AS coluna, CAST(`ind_bloq_preco` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_preco` UNION ALL SELECT 'ind_bloq_montante_icone' AS coluna, CAST(`ind_bloq_montante_icone` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_montante_icone` UNION ALL SELECT 'ind_bloq_montante_1' AS coluna, CAST(`ind_bloq_montante_1` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_montante_1` UNION ALL SELECT 'ind_bloq_montante_2' AS coluna, CAST(`ind_bloq_montante_2` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_montante_2` UNION ALL SELECT 'ind_bloq_manual_icone' AS coluna, CAST(`ind_bloq_manual_icone` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `ind_bloq_manual_icone`) SELECT coluna, valor, linhas FROM a ORDER BY coluna, linhas DESC;


## 10. Perfil numérico

Perfila quantidades e valores. Campos de códigos numéricos não são tratados como montantes.


In [0]:
SELECT * FROM (SELECT stack(9, 'qt_dias_desconto_1', CAST(n0_min AS DOUBLE), CAST(n0_max AS DOUBLE), CAST(n0_avg AS DOUBLE), CAST(n0_med AS DOUBLE), n0_neg, n0_zero, 'qt_pedido', CAST(n1_min AS DOUBLE), CAST(n1_max AS DOUBLE), CAST(n1_avg AS DOUBLE), CAST(n1_med AS DOUBLE), n1_neg, n1_zero, 'vl_montante', CAST(n2_min AS DOUBLE), CAST(n2_max AS DOUBLE), CAST(n2_avg AS DOUBLE), CAST(n2_med AS DOUBLE), n2_neg, n2_zero, 'vl_diferenca', CAST(n3_min AS DOUBLE), CAST(n3_max AS DOUBLE), CAST(n3_avg AS DOUBLE), CAST(n3_med AS DOUBLE), n3_neg, n3_zero, 'qt_em_upp', CAST(n4_min AS DOUBLE), CAST(n4_max AS DOUBLE), CAST(n4_avg AS DOUBLE), CAST(n4_med AS DOUBLE), n4_neg, n4_zero, 'qt_diferenca', CAST(n5_min AS DOUBLE), CAST(n5_max AS DOUBLE), CAST(n5_avg AS DOUBLE), CAST(n5_med AS DOUBLE), n5_neg, n5_zero, 'qt_diferenca_1', CAST(n6_min AS DOUBLE), CAST(n6_max AS DOUBLE), CAST(n6_avg AS DOUBLE), CAST(n6_med AS DOUBLE), n6_neg, n6_zero, 'qt_dias_liquido', CAST(n7_min AS DOUBLE), CAST(n7_max AS DOUBLE), CAST(n7_avg AS DOUBLE), CAST(n7_med AS DOUBLE), n7_neg, n7_zero, 'qt_dias_desconto_2', CAST(n8_min AS DOUBLE), CAST(n8_max AS DOUBLE), CAST(n8_avg AS DOUBLE), CAST(n8_med AS DOUBLE), n8_neg, n8_zero) AS (coluna, minimo, maximo, media, mediana_aproximada, negativos, zeros) FROM (SELECT MIN(`qt_dias_desconto_1`) AS n0_min, MAX(`qt_dias_desconto_1`) AS n0_max, AVG(`qt_dias_desconto_1`) AS n0_avg, PERCENTILE_APPROX(`qt_dias_desconto_1`, 0.5) AS n0_med, COUNT_IF(`qt_dias_desconto_1` < 0) AS n0_neg, COUNT_IF(`qt_dias_desconto_1` = 0) AS n0_zero, MIN(`qt_pedido`) AS n1_min, MAX(`qt_pedido`) AS n1_max, AVG(`qt_pedido`) AS n1_avg, PERCENTILE_APPROX(`qt_pedido`, 0.5) AS n1_med, COUNT_IF(`qt_pedido` < 0) AS n1_neg, COUNT_IF(`qt_pedido` = 0) AS n1_zero, MIN(`vl_montante`) AS n2_min, MAX(`vl_montante`) AS n2_max, AVG(`vl_montante`) AS n2_avg, PERCENTILE_APPROX(`vl_montante`, 0.5) AS n2_med, COUNT_IF(`vl_montante` < 0) AS n2_neg, COUNT_IF(`vl_montante` = 0) AS n2_zero, MIN(`vl_diferenca`) AS n3_min, MAX(`vl_diferenca`) AS n3_max, AVG(`vl_diferenca`) AS n3_avg, PERCENTILE_APPROX(`vl_diferenca`, 0.5) AS n3_med, COUNT_IF(`vl_diferenca` < 0) AS n3_neg, COUNT_IF(`vl_diferenca` = 0) AS n3_zero, MIN(`qt_em_upp`) AS n4_min, MAX(`qt_em_upp`) AS n4_max, AVG(`qt_em_upp`) AS n4_avg, PERCENTILE_APPROX(`qt_em_upp`, 0.5) AS n4_med, COUNT_IF(`qt_em_upp` < 0) AS n4_neg, COUNT_IF(`qt_em_upp` = 0) AS n4_zero, MIN(`qt_diferenca`) AS n5_min, MAX(`qt_diferenca`) AS n5_max, AVG(`qt_diferenca`) AS n5_avg, PERCENTILE_APPROX(`qt_diferenca`, 0.5) AS n5_med, COUNT_IF(`qt_diferenca` < 0) AS n5_neg, COUNT_IF(`qt_diferenca` = 0) AS n5_zero, MIN(`qt_diferenca_1`) AS n6_min, MAX(`qt_diferenca_1`) AS n6_max, AVG(`qt_diferenca_1`) AS n6_avg, PERCENTILE_APPROX(`qt_diferenca_1`, 0.5) AS n6_med, COUNT_IF(`qt_diferenca_1` < 0) AS n6_neg, COUNT_IF(`qt_diferenca_1` = 0) AS n6_zero, MIN(`qt_dias_liquido`) AS n7_min, MAX(`qt_dias_liquido`) AS n7_max, AVG(`qt_dias_liquido`) AS n7_avg, PERCENTILE_APPROX(`qt_dias_liquido`, 0.5) AS n7_med, COUNT_IF(`qt_dias_liquido` < 0) AS n7_neg, COUNT_IF(`qt_dias_liquido` = 0) AS n7_zero, MIN(`qt_dias_desconto_2`) AS n8_min, MAX(`qt_dias_desconto_2`) AS n8_max, AVG(`qt_dias_desconto_2`) AS n8_avg, PERCENTILE_APPROX(`qt_dias_desconto_2`, 0.5) AS n8_med, COUNT_IF(`qt_dias_desconto_2` < 0) AS n8_neg, COUNT_IF(`qt_dias_desconto_2` = 0) AS n8_zero FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) a) n ORDER BY coluna;


## 11. Totais para conciliação

Soma separadamente cada quantidade e valor. Totais DOUBLE são aproximados; comparar moedas e granularidade antes de conciliar.


In [0]:
SELECT coluna, total FROM (SELECT stack(9, 'qt_dias_desconto_1', CAST(s0 AS DOUBLE), 'qt_pedido', CAST(s1 AS DOUBLE), 'vl_montante', CAST(s2 AS DOUBLE), 'vl_diferenca', CAST(s3 AS DOUBLE), 'qt_em_upp', CAST(s4 AS DOUBLE), 'qt_diferenca', CAST(s5 AS DOUBLE), 'qt_diferenca_1', CAST(s6 AS DOUBLE), 'qt_dias_liquido', CAST(s7 AS DOUBLE), 'qt_dias_desconto_2', CAST(s8 AS DOUBLE)) AS (coluna, total) FROM (SELECT SUM(`qt_dias_desconto_1`) AS s0, SUM(`qt_pedido`) AS s1, SUM(`vl_montante`) AS s2, SUM(`vl_diferenca`) AS s3, SUM(`qt_em_upp`) AS s4, SUM(`qt_diferenca`) AS s5, SUM(`qt_diferenca_1`) AS s6, SUM(`qt_dias_liquido`) AS s7, SUM(`qt_dias_desconto_2`) AS s8 FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) a) s ORDER BY coluna;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dt_base_prazo_pgto' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_base_prazo_pgto` IS NULL) AS nulos, COUNT_IF(TRIM(`dt_base_prazo_pgto`) = '') AS vazios, COUNT_IF(TRIM(`dt_base_prazo_pgto`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dt_base_prazo_pgto`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dt_base_prazo_pgto`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dt_base_prazo_pgto`) AS menor_literal, MAX(`dt_base_prazo_pgto`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'dt_lancamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_lancamento` IS NULL) AS nulos, COUNT_IF(TRIM(`dt_lancamento`) = '') AS vazios, COUNT_IF(TRIM(`dt_lancamento`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dt_lancamento`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dt_lancamento`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dt_lancamento`) AS menor_literal, MAX(`dt_lancamento`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'dt_conversao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_conversao` IS NULL) AS nulos, COUNT_IF(TRIM(`dt_conversao`) = '') AS vazios, COUNT_IF(TRIM(`dt_conversao`) = '00000000') AS sentinela_00000000, COUNT_IF(TRIM(`dt_conversao`) = '9999-12-31') AS sentinela_9999_12_31, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_dd_mm_aaaa, COUNT_IF(TRIM(`dt_conversao`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, MIN(`dt_conversao`) AS menor_literal, MAX(`dt_conversao`) AS maior_literal FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'num_exercicio' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_exercicio`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_exercicio`))) AS tamanho_max, COUNT_IF(TRIM(`num_exercicio`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_exercicio`) RLIKE '^[0-9]+$' THEN TRIM(`num_exercicio`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_exercicio`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_exercicio`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_exerc_periodo_atual' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_exerc_periodo_atual`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_exerc_periodo_atual`))) AS tamanho_max, COUNT_IF(TRIM(`num_exerc_periodo_atual`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_exerc_periodo_atual`) RLIKE '^[0-9]+$' THEN TRIM(`num_exerc_periodo_atual`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_exerc_periodo_atual`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_exerc_periodo_atual`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_empresa`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_empresa`))) AS tamanho_max, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN TRIM(`cod_empresa`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_empresa`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_doc_faturamento' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_doc_faturamento`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_doc_faturamento`))) AS tamanho_max, COUNT_IF(TRIM(`num_doc_faturamento`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN TRIM(`num_doc_faturamento`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_faturamento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_faturamento`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'cod_emissor_fatura' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_emissor_fatura`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_emissor_fatura`))) AS tamanho_max, COUNT_IF(TRIM(`cod_emissor_fatura`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_emissor_fatura`) RLIKE '^[0-9]+$' THEN TRIM(`cod_emissor_fatura`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_emissor_fatura`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_emissor_fatura`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_doc_referencia' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_doc_referencia`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_doc_referencia`))) AS tamanho_max, COUNT_IF(TRIM(`num_doc_referencia`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_referencia`) RLIKE '^[0-9]+$' THEN TRIM(`num_doc_referencia`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_referencia`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_referencia`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_doc_compras' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_doc_compras`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_doc_compras`))) AS tamanho_max, COUNT_IF(TRIM(`num_doc_compras`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_compras`) RLIKE '^[0-9]+$' THEN TRIM(`num_doc_compras`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_doc_compras`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_doc_compras`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'cod_centro' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_centro`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_centro`))) AS tamanho_max, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_centro`) RLIKE '^[0-9]+$' THEN TRIM(`cod_centro`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_centro`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_centro`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'cod_material' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_material`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_material`))) AS tamanho_max, COUNT_IF(TRIM(`cod_material`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_material`) RLIKE '^[0-9]+$' THEN TRIM(`cod_material`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_material`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_material`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` UNION ALL SELECT 'num_item_pedido' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_item_pedido`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_item_pedido`))) AS tamanho_max, COUNT_IF(TRIM(`num_item_pedido`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_item_pedido`) RLIKE '^[0-9]+$' THEN TRIM(`num_item_pedido`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_item_pedido`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_item_pedido`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`num_exercicio`, t.`num_exerc_periodo_atual`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_doc_faturamento`, t.`cod_emissor_fatura`, t.`dt_base_prazo_pgto`, t.`dt_lancamento`, t.`qt_dias_desconto_1`, t.`num_doc_referencia`, t.`num_doc_compras`, t.`cod_centro`, t.`cod_material`, t.`num_item_pedido`, t.`cod_grupo_compradores`, t.`qt_pedido`, t.`sg_um_pedido`, t.`vl_montante`, t.`sg_moeda`, t.`vl_diferenca`, t.`sg_moeda_interna`, t.`tp_documento`, t.`cod_bloq_pagamento_log`, t.`nm_usuario`, t.`num_item_fatura`, t.`num_seq_classcont`, t.`num_item_doc_referencia`, t.`cod_bloqueio_pgto`, t.`sg_upp`, t.`tp_condicao`, t.`pct_desconto_2`, t.`pct_desconto_1`, t.`vl_taxa_cambio`, t.`st_fatura`, t.`qt_em_upp`, t.`qt_diferenca`, t.`qt_diferenca_1`, t.`cod_determ_preco`, t.`cod_pais_destino`, t.`cod_pais_procedencia`, t.`ind_nota_existente`, t.`num_pacote`, t.`ind_bloq_qtd`, t.`ind_bloq_qualidade_icone`, t.`ind_bloq_qtd_icone`, t.`ind_bloq_qualidade`, t.`ind_bloq_preco_icone`, t.`ind_bloq_qupp_icone`, t.`ind_bloq_qupp`, t.`ind_bloq_preco`, t.`ind_bloq_montante_icone`, t.`ind_bloq_montante_1`, t.`ind_bloq_montante_2`, t.`ind_bloq_manual_icone`, t.`ind_bloq_manual_1`, t.`ind_bloq_manual_2`, t.`ind_bloq_data_icone`, t.`ind_bloq_data`, t.`num_linha_servico`, t.`num_item_contrato_central`, t.`num_item_contrato_superior`, t.`qt_diferenca_dias`, t.`qt_dias_liquido`, t.`qt_dias_desconto_2`, t.`ind_debito_posterior`, t.`dt_conversao`, t.`num_contrato_central`, t.`num_contrato_basico`, t.`ind_custo_acessorio`, t.`cod_chave_controle_sasq`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `num_exercicio`, `num_exerc_periodo_atual`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `num_doc_referencia`, `num_doc_compras`, `cod_centro`, `cod_material`, `num_item_pedido`, `cod_grupo_compradores`, `qt_pedido`, `sg_um_pedido`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_centro`, `cod_empresa`, `cod_emissor_fatura`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_centro`, `cod_empresa`, `cod_emissor_fatura` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Não há coluna de carga documentada; registrar a data e hora de execução da consulta.


In [0]:
SELECT CURRENT_TIMESTAMP() AS instante_da_consulta, 'Sem coluna de carga no schema' AS limitacao;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Tipos de bloqueio e diferença


In [0]:
SELECT `cod_centro`, COUNT(*) AS linhas, COUNT_IF(`ind_bloq_qtd` IS NOT NULL AND TRIM(`ind_bloq_qtd`) <> '') AS bloq_qtd, COUNT_IF(`ind_bloq_preco` IS NOT NULL AND TRIM(`ind_bloq_preco`) <> '') AS bloq_preco, COUNT_IF(`vl_diferenca` IS NULL) AS diferenca_nula, SUM(`vl_montante`) AS montante_total FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `cod_centro` ORDER BY linhas DESC;


### 18.2. Cobertura cruzada das duas views MRBR

Compara apenas identificadores documentados em ambos os DESCRIBEs; a ausência de chave em uma view não define qual é oficial.


In [0]:
WITH a AS (SELECT num_exercicio, cod_empresa, num_doc_faturamento FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloq_pagamento_man` GROUP BY num_exercicio, cod_empresa, num_doc_faturamento), b AS (SELECT num_exercicio, cod_empresa, num_doc_faturamento FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY num_exercicio, cod_empresa, num_doc_faturamento) SELECT COUNT(*) AS chaves_uniao, COUNT_IF(a.num_doc_faturamento IS NOT NULL AND b.num_doc_faturamento IS NOT NULL) AS em_ambas, COUNT_IF(a.num_doc_faturamento IS NOT NULL AND b.num_doc_faturamento IS NULL) AS apenas_mrbr_1, COUNT_IF(a.num_doc_faturamento IS NULL AND b.num_doc_faturamento IS NOT NULL) AS apenas_mrbr_2 FROM a FULL OUTER JOIN b ON a.num_exercicio <=> b.num_exercicio AND a.cod_empresa <=> b.cod_empresa AND a.num_doc_faturamento <=> b.num_doc_faturamento;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `num_exercicio`, `num_exerc_periodo_atual`, `cod_empresa`, `num_doc_faturamento`, `cod_emissor_fatura`, `dt_base_prazo_pgto`, `dt_lancamento`, `qt_dias_desconto_1`, `num_doc_referencia`, `num_doc_compras`, `cod_centro`, `cod_material`, `num_item_pedido`, `cod_grupo_compradores`, `qt_pedido`, `sg_um_pedido`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`nm_usuario` AS STRING)),1,8) AS `nm_usuario_hash` FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` LIMIT 500;


## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios` GROUP BY `num_doc_faturamento`, `num_exercicio`, `num_item_fatura`), v AS (SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`vw_ds_pro_mrbr_bloqueado_devido_desvios`) SELECT 'MRBR' AS transacao, 'vw_ds_pro_mrbr_bloqueado_devido_desvios' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, v.centros_distintos, 'Chave candidata: num_doc_faturamento + num_exercicio + num_item_fatura' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
